**DATA READING**

In [0]:
df = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://bronze@carsaledatalake.dfs.core.windows.net/RawData")

In [0]:
df.display()

**DATA TRANSFORMATION**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = df.withColumn("Model_Category", split(col("Model_ID"),"-")[0])

In [0]:
df.display()

In [0]:
df = df.withColumn("Units_Sold", col('Units_Sold').cast(StringType()))

In [0]:
df.printSchema()

In [0]:
#arithmetic operation between Revenue and Units_Sold, we have a average revenue, i want price of per unit sold

df = df.withColumn("RevenuePerUnit", col('Revenue')/col('Units_Sold'))
df.display()

In [0]:
df = df.drop("RevPerUnit")

In [0]:
df.display()

**AD-HOC**

In [0]:
#how many units sold from each branch and year 
df_analysis = df.groupBy("year","BranchName").agg(sum("Units_sold").alias("TotalUnitsSold")).sort(['year','TotalUnitsSold'],ascending=[1,0])
df_analysis.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("Parquet").mode("overwrite").option("path","abfss://silver@carsaledatalake.dfs.core.windows.net/CarSales").save()